[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/225/blob/main/08_data_cleaning/08_6_Regex.ipynb)

# 08.6: Regular Expressions for Data Cleaning

In notebook 08.5 you used simple patterns like `r'(\w+)\.'` to extract a title from a name. That pattern is a regular expression. You wrote it without much explanation of the syntax because the pattern was short and the meaning was clear from context.

This notebook goes deeper into regular expressions, but always in service of a concrete cleaning problem. Every pattern introduced here solves a real data issue: normalizing phone numbers in multiple formats, stripping currency symbols, and parsing structured text. The goal is not to master regex as a language but to recognize the half-dozen patterns that come up repeatedly in data cleaning and know how to apply them. Let's load the data.

In [1]:
import pandas as pd

url = "https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_titanic.csv"
df = pd.read_csv(url)
df.columns = ["survived", "pclass", "name", "sex", "age", "sibsp", "parch", "fare"]

## The metacharacters you actually need

A regular expression is a string that describes a pattern. Most characters match themselves literally. A few characters have special meaning:

| Symbol | Meaning | Example |
|---|---|---|
| `\d` | Any digit 0-9 | `\d\d\d` matches `"404"` |
| `\w` | Any word character (letter, digit, underscore) | `\w+` matches `"hello_99"` |
| `\s` | Any whitespace (space, tab, newline) | `\s+` matches `"   "` |
| `.` | Any single character except newline | `a.c` matches `"abc"` or `"a1c"` |
| `+` | One or more of the preceding | `\d+` matches `"42"` or `"1000"` |
| `*` | Zero or more of the preceding | `\d*` matches `""` or `"42"` |
| `?` | Zero or one of the preceding | `colou?r` matches `"color"` or `"colour"` |
| `{n}` | Exactly n of the preceding | `\d{3}` matches exactly three digits |
| `[...]` | Any one character listed inside | `[aeiou]` matches one vowel |
| `[^...]` | Any character NOT listed | `[^\d]` matches anything that is not a digit |
| `^` | Start of string | `^Mr` matches strings that begin with `Mr` |
| `$` | End of string | `\d$` matches strings that end with a digit |
| `(...)` | Capture group | `(\d+)` captures one or more digits |
| `(?P<name>...)` | Named capture group | `(?P<code>\d+)` captures and names the group |

Each pattern below introduces only the metacharacters it needs.

## Problem 1: Phone numbers in multiple formats

A contact list might have phone numbers entered in any of these formats:
- `"(555) 867-5309"`: area code in parentheses
- `"555.867.5309"`: dots as separators
- `"555-867-5309"`: dashes as separators
- `"5558675309"`: all digits, no separators

These are all the same number. Before you can compare them, sort them, or validate them, you need to normalize them to a single format: ten consecutive digits.

In [2]:
phones = pd.Series([
    "(555) 867-5309",
    "555.867.5309",
    "555-867-5309",
    "5558675309",
    "(800) 555-1234",
    "not a phone"
])

# Remove every character that is NOT a digit
# [^\d] means "any character that is not a digit"
digits_only = phones.str.replace(r'[^\d]', '', regex=True)
print(digits_only.tolist())

['5558675309', '5558675309', '5558675309', '5558675309', '8005551234', '']


`[^\d]` is a negated character class: it matches any single character that is not a digit. Replacing every such character with an empty string strips out all punctuation and spaces, leaving only the raw digits.

Now all valid phone numbers are in the same format. The last entry (`"not a phone"`) produced an empty string because it has no digits at all. After normalization, you can validate which entries are actually phone numbers.

In [3]:
# Validate: a US phone number should be exactly 10 digits
# ^\d{10}$ means: start, exactly 10 digits, end
is_valid = digits_only.str.contains(r'^\d{10}$', regex=True)

result = pd.DataFrame({"original": phones, "digits": digits_only, "valid": is_valid})
result

,original,digits,valid
0,(555) 867-5309,5558675309,True
1,555.867.5309,5558675309,True
2,555-867-5309,5558675309,True
3,5558675309,5558675309,True
4,(800) 555-1234,8005551234,True
5,not a phone,,False


`^\d{10}$` anchors the pattern to the full string: `^` requires the match to start at the beginning, `\d{10}` requires exactly ten digits, and `$` requires the match to end at the end of the string. Without the anchors, a 12-digit string would also match because it contains ten digits somewhere inside it.

The `"not a phone"` entry is correctly flagged as invalid. All five numeric phone entries are valid.

## Problem 2: Currency amounts stored as strings

Suppose a dataset has a fare column where the values were exported from a spreadsheet and include currency symbols: `"$71.28"`, `"£7.25"`, `"22.00"`. You need to convert these to numeric values, but `pd.to_numeric()` will fail on the entries that have symbols.

In [4]:
fares_raw = pd.Series(["$71.28", "£7.25", "22.00", "$512.33", "8.05", "$0.00"])

# This fails because of the currency symbols
try:
    pd.to_numeric(fares_raw)
except ValueError as e:
    print("Error:", e)

Error: Unable to parse string "$71.28" at position 0


The fix is to strip any character that is not a digit or a decimal point, then convert.

In [5]:
# [^\d.] matches any character that is not a digit or a period
fares_clean = fares_raw.str.replace(r'[^\d.]', '', regex=True)
fares_numeric = pd.to_numeric(fares_clean)

pd.DataFrame({"raw": fares_raw, "cleaned": fares_clean, "numeric": fares_numeric})

,raw,cleaned,numeric
0,$71.28,71.28,71.28
1,£7.25,7.25,7.25
2,22.00,22.00,22.00
3,$512.33,512.33,512.33
4,8.05,8.05,8.05
5,$0.00,0.00,0.00


`[^\d.]` means "any character that is not a digit and is not a period." After replacing those characters with an empty string, every value is a plain decimal number that `pd.to_numeric()` can parse without error.

Stripping non-numeric characters with regex and then calling `pd.to_numeric()` handles most currency-cleaning situations regardless of which symbol is used.

## Problem 3: Extracting structured text with named groups

The Titanic name column uses a consistent structure that can be parsed with a single regex. This is a more complex version of what you saw in notebook 08.5: using named capture groups to extract multiple pieces at once.

The structure is `"Title. FirstName [MiddleName] LastName"`. To extract both the title and the last name (the last space-separated word), the pattern needs two capture groups.

In [6]:
# Pattern breakdown:
# ^(?P<title>\w+)\.  -> title at start, followed by period
# \s+                -> one or more spaces
# .+\s               -> any characters (the middle names), ending with a space
# (?P<last>\w+)$     -> last word before the end of the string
extracted = df["name"].str.extract(
    r'^(?P<title>\w+)\.\s+.+\s(?P<last>\w+)$'
)
extracted.head(8)

,title,last
0,Mr,Braund
1,Mrs,Cumings
2,Miss,Heikkinen
3,Mrs,Futrelle
4,Mr,Allen
5,Mr,Moran
6,Mr,McCarthy
7,Master,Palsson


The pattern anchors to the start with `^` and the end with `$`. The `.+\s` in the middle is a greedy match that consumes as many characters as possible while still leaving room for the last word. The `(?P<last>\w+)$` group picks up whatever word is left just before the end of the string.

The result is a DataFrame with two columns, named `title` and `last`, ready to use directly.

In [7]:
df["title"] = extracted["title"]
df["last_name"] = extracted["last"]

# Confirm the extraction worked on a few tricky names
df.loc[df["name"].str.contains(r'\('), ["name", "title", "last_name"]].head(5)

,name,title,last_name
1,Mrs. John Bradley (Florence Briggs Thayer) Cum...,Mrs,Cumings
3,Mrs. Jacques Heath (Lily May Peel) Futrelle,Mrs,Futrelle
8,Mrs. Oscar W (Elisabeth Vilhelmina Berg) Johnson,Mrs,Johnson
9,Mrs. Nicholas (Adele Achem) Nasser,Mrs,Nasser
15,Mrs. (Mary D Kingcome) Hewlett,Mrs,Hewlett


The names with parenthetical content, like `"Mrs. John Bradley (Florence Briggs Thayer) Cumings"`, are handled correctly: the `.+\s` middle section consumed everything including the parenthetical, and the last word before the end of the string is the actual last name.

But did the pattern work for *every* name? `str.extract()` does not raise an error when a string fails to match; it quietly returns NaN for that row. After any extraction, check for nulls before trusting the result.

In [8]:
print("Rows where extraction failed:", df["last_name"].isnull().sum())
df.loc[df["last_name"].isnull(), "name"].head(5)

Rows where extraction failed: 22


28                              Miss. Ellen O'Dwyer
39                       Miss. Jamila Nicola-Yarred
46                         Miss. Bridget O'Driscoll
48     Mrs. Josef (Josefine Franchi) Arnold-Franchi
124                     Master. Elias Nicola-Yarred
Name: name, dtype: str

Twenty-two names failed to match, all for the same reason: `\w` matches letters, digits, and underscores, but not hyphens or apostrophes. A last name like `O'Dwyer` or `Nicola-Yarred` cannot satisfy `(?P<last>\w+)$`, so the whole pattern fails and the row gets NaN. This is the most important habit to take from this notebook: a regex that works on the rows you looked at can still fail on rows you did not, and the failure is silent.

> The fix is to widen the character class: `(?P<last>[\w'-]+)$` accepts word characters plus apostrophes and hyphens. Widening a pattern one verified failure at a time is the normal rhythm of regex work.

## Filtering with `str.contains(regex=True)`

`str.contains()` accepts regex patterns, not just plain strings. This lets you filter rows based on complex conditions that a simple substring check cannot express.

In [9]:
# Find passengers whose last name ends with a vowel
# [aeiou]$ means: a vowel at the end of the string (case-insensitive)
vowel_endings = df[df["last_name"].str.contains(r'[aeiou]$', case=False, na=False)]
print(f"Passengers with last name ending in a vowel: {len(vowel_endings)}")

# Find passengers whose last name starts and ends with the same letter (just a demonstration)
# This shows str.contains is for filtering; complex logic like this is unusual in cleaning
print("\nSample vowel-ending last names:")
vowel_endings["last_name"].value_counts().head(8)

Passengers with last name ending in a vowel: 133

Sample vowel-ending last names:


last_name
Sage       7
Panula     6
Rice       5
Fortune    4
Lefebre    4
Baclini    4
Planke     3
Laroche    3
Name: count, dtype: int64

`case=False` makes the match case-insensitive. The `na=False` argument treats null values as non-matching instead of raising an error. These two arguments are good defaults for any `str.contains()` call on a column that might have missing values or inconsistent capitalization.

## The four regex patterns that cover most cleaning tasks

You will reach for the same patterns repeatedly. Here they are in one place:

| Task | Pattern | Call |
|---|---|---|
| Remove non-digit characters | `r'[^\d]'` | `str.replace(r'[^\d]', '', regex=True)` |
| Remove non-numeric characters (keep period) | `r'[^\d.]'` | `str.replace(r'[^\d.]', '', regex=True)` |
| Validate fixed-length digit string | `r'^\d{n}$'` | `str.contains(r'^\d{10}$', regex=True)` |
| Extract the first word | `r'^(\w+)'` | `str.extract(r'^(\w+)')` |
| Extract a word followed by a period | `r'(\w+)\.'` | `str.extract(r'(\w+)\.')` |

For anything more complex, the [Python `re` module documentation](https://docs.python.org/3/library/re.html) and [regex101.com](https://regex101.com) (a live pattern tester) are the right tools.

## What's next

You have now covered the full string-cleaning toolkit. Notebook 08.7 shifts to a different kind of data: dates and times. Almost every real-world dataset has a date column, and almost every CSV stores it as a string. You will learn to parse those strings into proper `datetime64` values, extract components like month and weekday, and compute durations by subtracting two timestamp columns.